# 🚗 Commuting Emissions Calculator
> **Quantify and reduce your organization's commuting CO₂ footprint using Google Maps API, Python, and BigQuery.**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/commuting-emissions-calculator/blob/main/commuting_emissions_colab.ipynb)

---

## What this notebook does

1. 📍 **Fetches real distances** from employee homes to workplace via Google Maps API
2. 💨 **Estimates CO₂ emissions** using Monte Carlo simulation over vehicle fleet emission factors
3. 🗺 **Generates an interactive map** color-coded by commute type
4. ☁️ **Uploads results to BigQuery** for historical tracking (optional)

### Research background
This tool is grounded in peer-reviewed research (under review, *Urban Climate*) showing that:
> Employees with long commutes (≥14.3 km) achieve **13.6× greater CO₂ savings** per teleworking day than short-distance commuters.

**Author:** Rafael Mesa-Manzano, PhD — University of Valencia, Spain

## 0. Install dependencies

In [ ]:
!pip install folium google-cloud-bigquery pyarrow db-dtypes --quiet

## 1. Configuration

Add your Google Maps API key to Colab Secrets (🔑 icon in the left sidebar) with the name `GOOGLE_MAPS_API_KEY`.

If you don't have a key yet, [get one here](https://developers.google.com/maps/documentation/distance-matrix/get-api-key). The Distance Matrix API has a free tier of $200/month.

In [1]:
from google.colab import userdata

# Load API key from Colab Secrets
try:
    GOOGLE_MAPS_API_KEY = userdata.get('GOOGLE_MAPS_API_KEY')
    print('✅ API key loaded from Colab Secrets')
except Exception:
    # Fallback: paste directly (not recommended for sharing)
    GOOGLE_MAPS_API_KEY = 'YOUR_API_KEY_HERE'
    print('⚠️  Using hardcoded API key — do not share this notebook publicly')

# Workplace address
WORKPLACE_ADDRESS = 'Universitat de València, Av. Blasco Ibáñez 13, Valencia, Spain'

print(f'Workplace: {WORKPLACE_ADDRESS}')

✅ API key loaded from Colab Secrets
Workplace: Universitat de València, Av. Blasco Ibáñez 13, Valencia, Spain


## 2. Load employee data

Upload your own CSV or use the sample data below.

**Required columns:** `employee_id`, `name`, `home_address`, `transport_mode`, `telework_days_per_week`

`transport_mode` accepts: `car`, `transit`, `bicycle`

In [2]:
import pandas as pd
import io

# Sample data — replace with your own CSV
SAMPLE_CSV = """employee_id,name,home_address,transport_mode,telework_days_per_week
E001,Ana García,Calle Mayor 10 Valencia Spain,car,2
E002,Carlos López,Avenida del Puerto 45 Valencia Spain,car,1
E003,María Fernández,Calle Colón 8 Valencia Spain,transit,3
E004,Juan Martínez,Paseo de la Alameda 22 Valencia Spain,car,0
E005,Laura Sánchez,Calle Sagunto 67 Valencia Spain,car,2
E006,Pedro Ruiz,Avenida Blasco Ibáñez 30 Valencia Spain,transit,1
E007,Isabel Torres,Calle Quart 15 Valencia Spain,car,3
E008,Antonio Moreno,Calle San Vicente 90 Valencia Spain,car,0
E009,Carmen Jiménez,Avenida Pérez Galdós 12 Valencia Spain,transit,2
E010,Francisco Navarro,Calle Guillem de Castro 55 Valencia Spain,car,1
E011,Rosa Díaz,Calle Turia 33 Valencia Spain,car,2
E012,Miguel Romero,Avenida Francia 18 Valencia Spain,car,1
E013,Elena Alonso,Calle Játiva 7 Valencia Spain,transit,0
E014,David Gutiérrez,Calle Ciscar 44 Valencia Spain,car,3
E015,Pilar Vázquez,Avenida Constitución 28 Valencia Spain,car,2
"""

employees = pd.read_csv(io.StringIO(SAMPLE_CSV))
print(f'✅ {len(employees)} employees loaded')
employees.head()

✅ 15 employees loaded


,employee_id,name,home_address,transport_mode,telework_days_per_week
0,E001,Ana García,Calle Mayor 10 Valencia Spain,car,2
1,E002,Carlos López,Avenida del Puerto 45 Valencia Spain,car,1
2,E003,María Fernández,Calle Colón 8 Valencia Spain,transit,3
3,E004,Juan Martínez,Paseo de la Alameda 22 Valencia Spain,car,0
4,E005,Laura Sánchez,Calle Sagunto 67 Valencia Spain,car,2


## 3. Fetch distances from Google Maps API

In [8]:
import requests
import time

def get_distances_debug(origins, destination, api_key):
    url = 'https://maps.googleapis.com/maps/api/distancematrix/json'
    params = {
        'origins': origins[0],  # solo uno para debug
        'destinations': destination,
        'mode': 'driving',
        'units': 'metric',
        'key': api_key
    }
    resp = requests.get(url, params=params, timeout=15).json()
    print(resp)  # ver respuesta completa
    return resp

get_distances_debug(employees['home_address'].tolist(), WORKPLACE_ADDRESS, GOOGLE_MAPS_API_KEY)

print('Fetching distances from Google Maps...')
distances = get_distances(employees['home_address'].tolist(), WORKPLACE_ADDRESS, GOOGLE_MAPS_API_KEY)
df = pd.concat([employees.reset_index(drop=True), distances], axis=1)
print(f'✅ Distances fetched for {distances["distance_km"].notna().sum()} employees')
print(distances)
print(distances.columns.tolist())
df[['name', 'distance_km', 'duration_min', 'transport_mode']].head(10)

{'destination_addresses': ['Av. de Blasco Ibáñez, 13, El Pla del Real, 46010 València, Valencia, Spain'], 'origin_addresses': ['C. Major, 10, 46920 Mislata, Valencia, Spain'], 'rows': [{'elements': [{'distance': {'text': '5.3 km', 'value': 5321}, 'duration': {'text': '15 mins', 'value': 887}, 'status': 'OK'}]}], 'status': 'OK'}
Fetching distances from Google Maps...
✅ Distances fetched for 15 employees
    distance_km  duration_min
0         5.321     14.783333
1         2.865      8.800000
2         3.744     12.100000
3         0.891      3.650000
4         1.805      7.550000
5         0.687      2.216667
6         3.224     10.533333
7         3.747     11.150000
8         4.922     12.516667
9         3.318      9.633333
10        3.034      9.816667
11        3.593     10.283333
12        3.783     11.100000
13        2.700      8.416667
14        2.305      8.433333
['distance_km', 'duration_min']


,name,distance_km,duration_min,transport_mode
0,Ana García,5.321,14.783333,car
1,Carlos López,2.865,8.800000,car
2,María Fernández,3.744,12.100000,transit
3,Juan Martínez,0.891,3.650000,car
4,Laura Sánchez,1.805,7.550000,car
5,Pedro Ruiz,0.687,2.216667,transit
6,Isabel Torres,3.224,10.533333,car
7,Antonio Moreno,3.747,11.150000,car
8,Carmen Jiménez,4.922,12.516667,transit
9,Francisco Navarro,3.318,9.633333,car


## 4. Calculate CO₂ emissions with Monte Carlo simulation

We simulate uncertainty in emission factors using 1,000 random draws from a normal distribution, based on European vehicle fleet data (EEA 2023).

| Mode | Mean (gCO₂/km) | Std |
|------|---------------|-----|
| Car | 158 | 32 |
| Transit | 68 | 12 |
| Bicycle | 0 | — |

In [9]:
import numpy as np

EMISSION_FACTORS = {
    'car':     {'mean': 158.0, 'std': 32.0},
    'transit': {'mean':  68.0, 'std': 12.0},
    'bicycle': {'mean':   0.0, 'std':  0.0},
}

WEEKS_PER_YEAR = 44
rng = np.random.default_rng(42)

def calc_emissions(row, n=1000):
    mode = row['transport_mode']
    dist = row['distance_km']
    telework = row['telework_days_per_week']
    if pd.isna(dist) or mode not in EMISSION_FACTORS:
        return pd.Series([None]*6)
    ef = EMISSION_FACTORS[mode]
    samples = np.clip(rng.normal(ef['mean'], ef['std'], n), 0, None)
    office_days = 5 - telework
    weekly_co2 = samples * dist * 2 * office_days
    return pd.Series({
        'co2_weekly_mean_g':  weekly_co2.mean(),
        'co2_weekly_p5_g':    np.percentile(weekly_co2, 5),
        'co2_weekly_p95_g':   np.percentile(weekly_co2, 95),
        'co2_annual_kg_mean': weekly_co2.mean() * WEEKS_PER_YEAR / 1000,
        'savings_1day_g':     (samples * dist * 2).mean(),
        'savings_2days_g':    (samples * dist * 4).mean(),
    })

def classify(d):
    if pd.isna(d): return 'unknown'
    if d <= 6.0:   return 'short'
    if d <= 14.3:  return 'medium'
    return 'long'

df['commute_type'] = df['distance_km'].apply(classify)
emissions = df.apply(calc_emissions, axis=1)
df = pd.concat([df, emissions], axis=1)

print('✅ Emissions calculated')
df[['name', 'distance_km', 'commute_type', 'co2_weekly_mean_g', 'co2_annual_kg_mean', 'savings_1day_g']].round(1)

✅ Emissions calculated


,name,distance_km,commute_type,co2_weekly_mean_g,co2_annual_kg_mean,savings_1day_g
0,Ana García,5.3,short,5014.8,220.7,1671.6
1,Carlos López,2.9,short,3561.7,156.7,890.4
2,María Fernández,3.7,short,1024.4,45.1,512.2
3,Juan Martínez,0.9,short,1407.4,61.9,281.5
4,Laura Sánchez,1.8,short,1703.7,75.0,567.9
5,Pedro Ruiz,0.7,short,377.0,16.6,94.2
6,Isabel Torres,3.2,short,2043.6,89.9,1021.8
7,Antonio Moreno,3.7,short,5868.9,258.2,1173.8
8,Carmen Jiménez,4.9,short,2014.4,88.6,671.5
9,Francisco Navarro,3.3,short,4158.8,183.0,1039.7


## 5. Summary by commute type

In [10]:
summary = df.groupby('commute_type').agg(
    employees=('employee_id', 'count'),
    avg_distance_km=('distance_km', 'mean'),
    avg_weekly_co2_g=('co2_weekly_mean_g', 'mean'),
    avg_annual_co2_kg=('co2_annual_kg_mean', 'mean'),
    avg_savings_1day_g=('savings_1day_g', 'mean'),
).round(1)

print('── SUMMARY BY COMMUTE TYPE ──')
display(summary)

total = df['co2_annual_kg_mean'].sum()
print(f'\n🌍 Total fleet annual emissions: {total:.1f} kg CO₂ ({total/1000:.2f} tonnes)')

── SUMMARY BY COMMUTE TYPE ──


,employees,avg_distance_km,avg_weekly_co2_g,avg_annual_co2_kg,avg_savings_1day_g
commute_type,,,,,
short,15,3.1,2734.6,120.3,807.3



🌍 Total fleet annual emissions: 1804.9 kg CO₂ (1.80 tonnes)


## 6. Interactive map

In [19]:
import folium
from folium.plugins import MarkerCluster
import polyline as pl
import numpy as np

COLORS = {'short': '#2ecc71', 'medium': '#f39c12', 'long': '#e74c3c', 'unknown': '#95a1ac'}

def geocode(address, api_key):
    r = requests.get('https://maps.googleapis.com/maps/api/geocode/json',
                     params={'address': address, 'key': api_key}, timeout=10).json()
    if r['status'] == 'OK':
        loc = r['results'][0]['geometry']['location']
        return loc['lat'], loc['lng']
    return None

def get_route(origin_coords, destination_coords, api_key):
    url = 'https://maps.googleapis.com/maps/api/directions/json'
    params = {
        'origin': f"{origin_coords[0]},{origin_coords[1]}",
        'destination': f"{destination_coords[0]},{destination_coords[1]}",
        'mode': 'driving',
        'key': api_key
    }
    r = requests.get(url, params=params, timeout=10).json()
    if r['status'] == 'OK':
        encoded = r['routes'][0]['overview_polyline']['points']
        return pl.decode(encoded)
    return None

def emissions_to_width(co2_g, min_co2, max_co2):
    """Escala las emisiones a un grosor de línea entre 2 y 10."""
    if max_co2 == min_co2:
        return 4
    return 2 + 8 * (co2_g - min_co2) / (max_co2 - min_co2)

def emissions_to_color(co2_g, min_co2, max_co2):
    """Verde → amarillo → rojo según nivel de emisiones."""
    if max_co2 == min_co2:
        return '#f39c12'
    ratio = (co2_g - min_co2) / (max_co2 - min_co2)
    if ratio < 0.5:
        # verde → amarillo
        r = int(255 * ratio * 2)
        g = 200
        b = 50
    else:
        # amarillo → rojo
        r = 220
        g = int(200 * (1 - (ratio - 0.5) * 2))
        b = 50
    return f'#{r:02x}{g:02x}{b:02x}'

print('Geocoding workplace...')
wp_coords = geocode(WORKPLACE_ADDRESS, GOOGLE_MAPS_API_KEY)

# Calcular rango de emisiones para escalar visualmente
valid_co2 = df['co2_weekly_mean_g'].dropna()
min_co2 = valid_co2.min()
max_co2 = valid_co2.max()
print(f'CO₂ range: {min_co2:.0f} g — {max_co2:.0f} g/week')

m = folium.Map(location=wp_coords, zoom_start=11, tiles='CartoDB positron')

# Workplace marker
folium.Marker(
    wp_coords,
    popup='<b>Workplace</b>',
    icon=folium.Icon(color='blue', icon='building', prefix='fa'),
    tooltip='Workplace'
).add_to(m)

print('Drawing emission-weighted routes...')
for _, row in df.iterrows():
    coords = geocode(row['home_address'], GOOGLE_MAPS_API_KEY)
    if not coords:
        continue

    co2 = row.get('co2_weekly_mean_g', 0) or 0
    color = emissions_to_color(co2, min_co2, max_co2)
    width = emissions_to_width(co2, min_co2, max_co2)

    # Ruta real con grosor y color proporcional a emisiones
    route_points = get_route(coords, wp_coords, GOOGLE_MAPS_API_KEY)
    if route_points:
        folium.PolyLine(
            locations=route_points,
            color=color,
            weight=width,
            opacity=0.8,
            tooltip=f"{row['name']} — {co2:,.0f} gCO₂/week"
        ).add_to(m)

    # Marker empleado
    popup_html = f"""
    <div style='font-family:sans-serif;font-size:13px;min-width:190px'>
        <b>{row['name']}</b><br>
        <hr style='margin:4px 0'>
        📍 <b>{row.get('distance_km', 0):.1f} km</b> | ⏱ <b>{row.get('duration_min', 0):.0f} min</b><br>
        🏷 Commute: <b>{row.get('commute_type','')}</b> | 🚗 <b>{row['transport_mode']}</b><br>
        <hr style='margin:4px 0'>
        💨 Weekly CO₂: <b>{co2:,.0f} g</b><br>
        📅 Annual CO₂: <b>{row.get('co2_annual_kg_mean', 0):.1f} kg</b><br>
        💚 Saving 1 telework day: <b>{row.get('savings_1day_g', 0):,.0f} g/week</b>
    </div>
    """
    folium.CircleMarker(
        coords,
        radius=7,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.9,
        popup=folium.Popup(popup_html, max_width=230),
        tooltip=f"{row['name']} — {co2:,.0f} gCO₂/week"
    ).add_to(m)

    time.sleep(0.1)

# Leyenda
legend = f"""
<div style='position:fixed;bottom:30px;left:30px;z-index:1000;background:white;
padding:12px 16px;border-radius:8px;box-shadow:0 2px 8px rgba(0,0,0,0.2);font-size:13px'>
<b>Weekly CO₂ emissions</b><br>
<span style='color:#00c832'>●</span> Low ({min_co2:.0f} g)<br>
<span style='color:#f39c12'>●</span> Medium<br>
<span style='color:#dc3232'>●</span> High ({max_co2:.0f} g)<br>
<br>
<b>Line width</b> = emission level<br>
<span style='color:#3498db'>■</span> Workplace
</div>"""
m.get_root().html.add_child(folium.Element(legend))

display(m)
m.save('emissions_map.html')
print('✅ Emission-weighted map saved')

Geocoding workplace...
CO₂ range: 377 g — 5869 g/week
Drawing emission-weighted routes...


✅ Emission-weighted map saved


## 7. Save results

In [12]:
df.to_csv('commuting_emissions.csv', index=False)
print('✅ Results saved to commuting_emissions.csv')

# Download to your computer
from google.colab import files
files.download('commuting_emissions.csv')
files.download('emissions_map.html')

✅ Results saved to commuting_emissions.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 8. Upload to BigQuery (optional)

Skip this cell if you don't have a GCP project.

In [14]:
from google.colab import auth
from google.cloud import bigquery
from datetime import datetime

GCP_PROJECT_ID = 'padel-453207'  # ← change this

auth.authenticate_user()
client = bigquery.Client(project=GCP_PROJECT_ID)

upload_df = df.copy()
upload_df['run_date'] = datetime.today().date().isoformat()

table_ref = f'{GCP_PROJECT_ID}.commuting_emissions.emissions_by_employee'
dataset = bigquery.Dataset(f'{GCP_PROJECT_ID}.commuting_emissions')
dataset.location = 'EU'
client.create_dataset(dataset, exists_ok=True)

job = client.load_table_from_dataframe(
    upload_df, table_ref,
    job_config=bigquery.LoadJobConfig(write_disposition='WRITE_APPEND')
)
job.result()
print(f'✅ {len(upload_df)} rows uploaded to BigQuery → {table_ref}')

✅ 15 rows uploaded to BigQuery → padel-453207.commuting_emissions.emissions_by_employee
